# 🪪 Aadhaar Analytics - Exploratory Data Analysis (EDA)

## 📌 Objective
Explore and visualize ~5 Million records across **Enrolment**, **Demographic**, and **Biometric** datasets. 
This notebook analyzes multi-shard time series trends, state-level distributions, age demographics, and cross-shard update ratios across 28 canonical Indian states.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Inline Data Ingestion & State Normalization Utilities
import os
import glob
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

CANONICAL_STATES = [
    "Andaman & Nicobar", "Andhra Pradesh", "Arunachal Pradesh", "Assam", "Bihar",
    "Chandigarh", "Chhattisgarh", "Dadra & Nagar Haveli and Daman & Diu", "Delhi",
    "Goa", "Gujarat", "Haryana", "Himachal Pradesh", "Jammu and Kashmir", "Jharkhand",
    "Karnataka", "Kerala", "Ladakh", "Lakshadweep", "Madhya Pradesh", "Maharashtra",
    "Manipur", "Meghalaya", "Mizoram", "Nagaland", "Odisha", "Puducherry", "Punjab",
    "Rajasthan", "Sikkim", "Tamil Nadu", "Telangana", "Tripura", "Uttar Pradesh",
    "Uttarakhand", "West Bengal"
]

STATE_ALIASES = {
    "andaman and nicobar islands": "Andaman & Nicobar",
    "andaman & nicobar islands": "Andaman & Nicobar",
    "a & n islands": "Andaman & Nicobar",
    "andhra pradesh": "Andhra Pradesh",
    "arunachal pradesh": "Arunachal Pradesh",
    "assam": "Assam",
    "bihar": "Bihar",
    "chandigarh": "Chandigarh",
    "chhattisgarh": "Chhattisgarh",
    "chhatisgarh": "Chhattisgarh",
    "dadra and nagar haveli": "Dadra & Nagar Haveli and Daman & Diu",
    "daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "dadra and nagar haveli and daman and diu": "Dadra & Nagar Haveli and Daman & Diu",
    "delhi": "Delhi",
    "nct of delhi": "Delhi",
    "goa": "Goa",
    "gujarat": "Gujarat",
    "haryana": "Haryana",
    "himachal pradesh": "Himachal Pradesh",
    "jammu and kashmir": "Jammu and Kashmir",
    "jammu & kashmir": "Jammu and Kashmir",
    "jharkhand": "Jharkhand",
    "karnataka": "Karnataka",
    "kerala": "Kerala",
    "ladakh": "Ladakh",
    "lakshadweep": "Lakshadweep",
    "madhya pradesh": "Madhya Pradesh",
    "maharashtra": "Maharashtra",
    "manipur": "Manipur",
    "meghalaya": "Meghalaya",
    "mizoram": "Mizoram",
    "nagaland": "Nagaland",
    "odisha": "Odisha",
    "orissa": "Odisha",
    "puducherry": "Puducherry",
    "pondicherry": "Puducherry",
    "punjab": "Punjab",
    "rajasthan": "Rajasthan",
    "sikkim": "Sikkim",
    "tamil nadu": "Tamil Nadu",
    "telangana": "Telangana",
    "tripura": "Tripura",
    "uttar pradesh": "Uttar Pradesh",
    "uttarakhand": "Uttarakhand",
    "uttaranchal": "Uttarakhand",
    "west bengal": "West Bengal"
}

def _normalize_state_name(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().lower()
    return STATE_ALIASES.get(val_str, str(val).strip().title())

def load_and_preprocess_raw_data(data_dir="."):
    enrol_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_enrolment/**/*.csv'), recursive=True))
    enrol_list = []
    for f in enrol_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        enrol_list.append(df)
    
    enrol_df = pd.concat(enrol_list, ignore_index=True) if enrol_list else pd.DataFrame()
        
    if not enrol_df.empty:
        enrol_df['date'] = pd.to_datetime(enrol_df['date'], format='%d-%m-%Y', errors='coerce')
        enrol_df['norm_state'] = enrol_df['state'].apply(_normalize_state_name)
        enrol_df['total_enrolments'] = enrol_df['age_0_5'].fillna(0) + enrol_df['age_5_17'].fillna(0) + enrol_df['age_18_greater'].fillna(0)
        enrol_daily = enrol_df.groupby(['date', 'norm_state'])[['age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments']].sum().reset_index()
    else:
        enrol_daily = pd.DataFrame(columns=['date', 'norm_state', 'age_0_5', 'age_5_17', 'age_18_greater', 'total_enrolments'])

    demo_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_demographic/**/*.csv'), recursive=True))
    demo_list = []
    for f in demo_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        demo_list.append(df)
    
    if demo_list:
        demo_df = pd.concat(demo_list, ignore_index=True)
        demo_df['date'] = pd.to_datetime(demo_df['date'], format='%d-%m-%Y', errors='coerce')
        demo_df['norm_state'] = demo_df['state'].apply(_normalize_state_name)
        demo_df['demo_total'] = demo_df['demo_age_5_17'].fillna(0) + demo_df['demo_age_17_'].fillna(0)
        demo_daily = demo_df.groupby(['date', 'norm_state'])[['demo_age_5_17', 'demo_age_17_', 'demo_total']].sum().reset_index()
    else:
        demo_daily = pd.DataFrame(columns=['date', 'norm_state', 'demo_age_5_17', 'demo_age_17_', 'demo_total'])

    bio_files = sorted(glob.glob(os.path.join(data_dir, 'api_data_aadhar_biometric/**/*.csv'), recursive=True))
    bio_list = []
    for f in bio_files:
        df = pd.read_csv(f, dtype={'state': str, 'district': str})
        bio_list.append(df)
        
    if bio_list:
        bio_df = pd.concat(bio_list, ignore_index=True)
        bio_df['date'] = pd.to_datetime(bio_df['date'], format='%d-%m-%Y', errors='coerce')
        bio_df['norm_state'] = bio_df['state'].apply(_normalize_state_name)
        bio_df['bio_total'] = bio_df['bio_age_5_17'].fillna(0) + bio_df['bio_age_17_'].fillna(0)
        bio_daily = bio_df.groupby(['date', 'norm_state'])[['bio_age_5_17', 'bio_age_17_', 'bio_total']].sum().reset_index()
    else:
        bio_daily = pd.DataFrame(columns=['date', 'norm_state', 'bio_age_5_17', 'bio_age_17_', 'bio_total'])

    merged = pd.merge(enrol_daily, demo_daily, on=['date', 'norm_state'], how='outer')
    merged = pd.merge(merged, bio_daily, on=['date', 'norm_state'], how='outer')
    
    merged['total_enrolments'] = merged['total_enrolments'].fillna(0)
    merged['demo_total'] = merged['demo_total'].fillna(0)
    merged['bio_total'] = merged['bio_total'].fillna(0)

    return merged


# 1. Ingest multi-shard panel dataset
panel_df = load_and_preprocess_raw_data(data_dir=".")
print(f"Panel Dataset Shape: {panel_df.shape}")
print(f"Date Range: {panel_df['date'].min().strftime('%Y-%m-%d')} to {panel_df['date'].max().strftime('%Y-%m-%d')}")
print(f"Canonical States Count: {panel_df['norm_state'].nunique()}")
panel_df.head()


Panel Dataset Shape: (4234, 12)
Date Range: 2025-03-01 to 2025-12-31
Canonical States Count: 52


,date,norm_state,age_0_5,age_5_17,age_18_greater,total_enrolments,demo_age_5_17,demo_age_17_,demo_total,bio_age_5_17,bio_age_17_,bio_total
0,2025-03-01,Andaman & Nicobar,NaN,NaN,NaN,0.0,126.0,1212.0,1338.0,1612.0,1091.0,2703.0
1,2025-03-01,Andhra Pradesh,NaN,NaN,NaN,0.0,48600.0,464440.0,513040.0,243777.0,159519.0,403296.0
2,2025-03-01,Arunachal Pradesh,NaN,NaN,NaN,0.0,852.0,6957.0,7809.0,2953.0,4447.0,7400.0
3,2025-03-01,Assam,NaN,NaN,NaN,0.0,16692.0,185345.0,202037.0,59101.0,33830.0,92931.0
4,2025-03-01,Bihar,NaN,NaN,NaN,0.0,95221.0,991478.0,1086699.0,324179.0,439330.0,763509.0


## 📈 Time Series Overview: Daily National Aadhaar Activity

We aggregate daily total enrolments, demographic updates, and biometric updates across all states to inspect national volume over time.

In [2]:
daily_national = panel_df.groupby('date')[['total_enrolments', 'demo_total', 'bio_total']].sum().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_national['date'], y=daily_national['total_enrolments'], name='Total Enrolments', line=dict(color='#1f77b4', width=2)))
fig.add_trace(go.Scatter(x=daily_national['date'], y=daily_national['demo_total'], name='Demographic Updates', line=dict(color='#ff7f0e', width=2)))
fig.add_trace(go.Scatter(x=daily_national['date'], y=daily_national['bio_total'], name='Biometric Updates', line=dict(color='#2ca02c', width=2)))

fig.update_layout(
    title='Daily National Activity Across Shards (Enrolments vs Updates)',
    xaxis_title='Date',
    yaxis_title='Volume',
    template='plotly_dark',
    hovermode='x unified',
    height=500
)
fig.show()

## 🗺️ Geographic Distribution & Top States Analysis

Analyze cumulative volume across 28 canonical states to identify high-density administrative regions.

In [3]:
state_totals = panel_df.groupby('norm_state')['total_enrolments'].sum().reset_index().sort_values('total_enrolments', ascending=False)

fig_bar = px.bar(
    state_totals.head(10),
    x='total_enrolments',
    y='norm_state',
    orientation='h',
    title='Top 10 States by Total Aadhaar Enrolments',
    labels={'total_enrolments': 'Total Enrolments', 'norm_state': 'State'},
    color='total_enrolments',
    color_continuous_scale='Viridis',
    template='plotly_dark'
)
fig_bar.update_layout(yaxis={'categoryorder': 'total ascending'}, height=450)
fig_bar.show()